In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")

nav.head()

In [2]:
nav.shape

NameError: name 'nav' is not defined

In [3]:
nav = pd.read_csv("../data/raw/02_nav_history.csv")

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [4]:
nav.shape

(46000, 3)

In [5]:
nav["date"] = pd.to_datetime(nav["date"])

nav = nav.sort_values(["amfi_code", "date"])

nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

In [6]:
nav.head(10)

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639
5756,100016,2022-01-11,513.5542,0.005562
5757,100016,2022-01-12,512.3195,-0.002404
5758,100016,2022-01-13,510.2445,-0.004050
5759,100016,2022-01-14,514.3636,0.008073


In [7]:
nav["daily_return"].describe()

count    45960.000000
mean         0.000631
std          0.010290
min         -0.058102
25%         -0.005042
50%          0.000340
75%          0.006324
max          0.064713
Name: daily_return, dtype: float64

In [8]:
cagr_data = []

for code in nav["amfi_code"].unique():

    temp = nav[nav["amfi_code"] == code]

    start_nav = temp.iloc[0]["nav"]
    end_nav = temp.iloc[-1]["nav"]

    years = (
        temp.iloc[-1]["date"] -
        temp.iloc[0]["date"]
    ).days / 365

    cagr = ((end_nav / start_nav) ** (1 / years) - 1) * 100

    cagr_data.append([code, cagr])

cagr_df = pd.DataFrame(
    cagr_data,
    columns=["amfi_code", "cagr_pct"]
)

cagr_df.head()

,amfi_code,cagr_pct
0,100016,2.635246
1,100025,4.455091
2,100033,30.099704
3,101206,23.520489
4,101207,7.933121


In [9]:
cagr_df.sort_values(
    "cagr_pct",
    ascending=False
).head(10)

,amfi_code,cagr_pct
25,120505,32.801599
21,119598,32.398084
39,149324,32.262108
36,148569,31.924486
34,148567,30.949920
30,120843,30.883326
2,100033,30.099704
38,149323,29.558105
16,119094,28.192608
19,119551,25.784921


In [10]:
sharpe_data = []

for code in nav["amfi_code"].unique():

    temp = nav[nav["amfi_code"] == code]

    mean_return = temp["daily_return"].mean()

    std_return = temp["daily_return"].std()

    sharpe = (mean_return / std_return) * np.sqrt(252)

    sharpe_data.append([code, sharpe])

sharpe_df = pd.DataFrame(
    sharpe_data,
    columns=["amfi_code", "sharpe_ratio"]
)

sharpe_df.head()

,amfi_code,sharpe_ratio
0,100016,0.245276
1,100025,1.097333
2,100033,1.436947
3,101206,1.473390
4,101207,0.414625


In [11]:
sharpe_df.sort_values(
    "sharpe_ratio",
    ascending=False
).head(10)

,amfi_code,sharpe_ratio
27,120507,13.655946
31,120844,12.573992
5,101208,12.019194
34,148567,1.906241
30,120843,1.715884
19,119551,1.681289
36,148569,1.602702
9,118632,1.541077
25,120505,1.517047
38,149323,1.498398


In [12]:
sortino_data = []

for code in nav["amfi_code"].unique():

    temp = nav[nav["amfi_code"] == code]

    mean_return = temp["daily_return"].mean()

    downside = temp[
        temp["daily_return"] < 0
    ]["daily_return"]

    downside_std = downside.std()

    sortino = (
        mean_return / downside_std
    ) * np.sqrt(252)

    sortino_data.append([code, sortino])

sortino_df = pd.DataFrame(
    sortino_data,
    columns=["amfi_code", "sortino_ratio"]
)

sortino_df.head()

,amfi_code,sortino_ratio
0,100016,0.427276
1,100025,1.822432
2,100033,2.403193
3,101206,2.581215
4,101207,0.705169


In [13]:
sortino_df.sort_values(
    "sortino_ratio",
    ascending=False
).head(10)

,amfi_code,sortino_ratio
27,120507,28.983395
31,120844,26.597183
5,101208,24.773822
34,148567,3.139985
30,120843,3.104586
19,119551,2.978156
36,148569,2.786282
9,118632,2.635947
25,120505,2.608779
24,120504,2.601129


In [14]:
maxdd_data = []

for code in nav["amfi_code"].unique():

    temp = nav[nav["amfi_code"] == code].copy()

    temp["running_max"] = temp["nav"].cummax()

    temp["drawdown"] = (
        temp["nav"] / temp["running_max"] - 1
    )

    max_dd = temp["drawdown"].min()

    maxdd_data.append([code, max_dd])

maxdd_df = pd.DataFrame(
    maxdd_data,
    columns=["amfi_code", "max_drawdown"]
)

maxdd_df.head()

,amfi_code,max_drawdown
0,100016,-0.247344
1,100025,-0.043083
2,100033,-0.162172
3,101206,-0.112916
4,101207,-0.354469


In [15]:
maxdd_df.sort_values(
    "max_drawdown"
).head(10)

,amfi_code,max_drawdown
22,119599,-0.525742
17,119095,-0.516778
4,101207,-0.354469
39,149324,-0.311719
21,119598,-0.287060
7,102886,-0.280011
0,100016,-0.247344
29,120842,-0.240035
11,118634,-0.233449
15,119093,-0.217514


In [16]:
performance_df = cagr_df.merge(
    sharpe_df,
    on="amfi_code"
)

performance_df = performance_df.merge(
    sortino_df,
    on="amfi_code"
)

performance_df = performance_df.merge(
    maxdd_df,
    on="amfi_code"
)

performance_df.head()

,amfi_code,cagr_pct,sharpe_ratio,sortino_ratio,max_drawdown
0,100016,2.635246,0.245276,0.427276,-0.247344
1,100025,4.455091,1.097333,1.822432,-0.043083
2,100033,30.099704,1.436947,2.403193,-0.162172
3,101206,23.520489,1.473390,2.581215,-0.112916
4,101207,7.933121,0.414625,0.705169,-0.354469


In [17]:
performance_df.shape

(40, 5)

In [18]:
performance_df["cagr_rank"] = performance_df[
    "cagr_pct"
].rank(ascending=False)

performance_df["sharpe_rank"] = performance_df[
    "sharpe_ratio"
].rank(ascending=False)

performance_df["sortino_rank"] = performance_df[
    "sortino_ratio"
].rank(ascending=False)

performance_df["drawdown_rank"] = performance_df[
    "max_drawdown"
].rank(ascending=False)

In [19]:
performance_df["fund_score"] = (
    0.40 * performance_df["cagr_rank"] +
    0.30 * performance_df["sharpe_rank"] +
    0.20 * performance_df["sortino_rank"] +
    0.10 * performance_df["drawdown_rank"]
)

In [20]:
performance_df.sort_values(
    "fund_score"
).head(10)

,amfi_code,cagr_pct,sharpe_ratio,sortino_ratio,max_drawdown,cagr_rank,sharpe_rank,sortino_rank,drawdown_rank,fund_score
34,148567,30.949920,1.906241,3.139985,-0.112657,5.0,4.0,4.0,8.0,4.8
30,120843,30.883326,1.715884,3.104586,-0.129740,6.0,5.0,5.0,13.0,6.2
36,148569,31.924486,1.602702,2.786282,-0.163967,4.0,7.0,7.0,21.0,7.2
25,120505,32.801599,1.517047,2.608779,-0.181885,1.0,9.0,9.0,25.0,7.4
19,119551,25.784921,1.681289,2.978156,-0.150124,10.0,6.0,6.0,17.0,8.7
9,118632,24.031196,1.541077,2.635947,-0.174141,11.0,8.0,8.0,23.0,10.7
38,149323,29.558105,1.498398,2.481754,-0.172481,8.0,10.0,12.0,22.0,10.8
2,100033,30.099704,1.436947,2.403193,-0.162172,7.0,13.0,13.0,20.0,11.3
3,101206,23.520489,1.473390,2.581215,-0.112916,12.0,12.0,11.0,9.0,11.5
24,120504,23.277448,1.479051,2.601129,-0.125883,13.0,11.0,10.0,12.0,11.7


In [21]:
performance_df.to_csv(
    "../data/processed/fund_scorecard.csv",
    index=False
)

In [22]:
import os

os.listdir("../data/raw")

['01_fund_master.csv',
 '02_nav_history.csv',
 '03_aum_by_fund_house.csv',
 '04_monthly_sip_inflows.csv',
 '05_category_inflows.csv',
 '06_industry_folio_count.csv',
 '07_scheme_performance.csv',
 '08_investor_transactions.csv',
 '09_portfolio_holdings.csv',
 '10_benchmark_indices.csv',
 'bluechip_nav_combined.csv',
 'hdfc_top100_nav.csv']

In [23]:
os.listdir("../data/raw")

['01_fund_master.csv',
 '02_nav_history.csv',
 '03_aum_by_fund_house.csv',
 '04_monthly_sip_inflows.csv',
 '05_category_inflows.csv',
 '06_industry_folio_count.csv',
 '07_scheme_performance.csv',
 '08_investor_transactions.csv',
 '09_portfolio_holdings.csv',
 '10_benchmark_indices.csv',
 'bluechip_nav_combined.csv',
 'hdfc_top100_nav.csv']

In [24]:
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

In [25]:
benchmark.head()

,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [26]:
benchmark.columns

Index(['date', 'index_name', 'close_value'], dtype='str')

In [27]:
benchmark["index_name"].unique()

<StringArray>
[        'NIFTY50',        'NIFTY100', 'NIFTY_MIDCAP150',    'BSE_SMALLCAP',
        'NIFTY500',   'CRISIL_LIQUID',     'CRISIL_GILT']
Length: 7, dtype: str

In [28]:
nifty100 = benchmark[
    benchmark["index_name"] == "NIFTY100"
].copy()

nifty100["date"] = pd.to_datetime(
    nifty100["date"]
)

nifty100 = nifty100.sort_values("date")

nifty100["benchmark_return"] = (
    nifty100["close_value"].pct_change()
)

nifty100.head()

,date,index_name,close_value,benchmark_return
1150,2022-01-03,NIFTY100,17778.24,NaN
1151,2022-01-04,NIFTY100,17537.52,-0.013540
1152,2022-01-05,NIFTY100,17607.73,0.004003
1153,2022-01-06,NIFTY100,17556.05,-0.002935
1154,2022-01-07,NIFTY100,17664.02,0.006150


In [29]:
from scipy.stats import linregress

In [30]:
alpha_beta_data = []

for code in nav["amfi_code"].unique():

    fund = nav[
        ["date", "daily_return"]
    ][nav["amfi_code"] == code]

    merged = pd.merge(
        fund,
        nifty100[
            ["date", "benchmark_return"]
        ],
        on="date",
        how="inner"
    )

    merged = merged.dropna()

    slope, intercept, r, p, std = linregress(
        merged["benchmark_return"],
        merged["daily_return"]
    )

    beta = slope

    alpha = intercept * 252 * 100

    alpha_beta_data.append(
        [code, alpha, beta]
    )

alpha_beta_df = pd.DataFrame(
    alpha_beta_data,
    columns=[
        "amfi_code",
        "alpha_pct",
        "beta"
    ]
)

alpha_beta_df.head()

,amfi_code,alpha_pct,beta
0,100016,3.747581,-0.058268
1,100025,4.281792,0.001158
2,100033,27.195355,0.005104
3,101206,21.399785,0.021086
4,101207,10.897092,-0.065289


In [ ]:
alpha_beta_data = []

for code in nav["amfi_code"].unique():

    fund = nav[
        ["date", "daily_return"]
    ][nav["amfi_code"] == code]

    merged = pd.merge(
        fund,
        nifty100[
            ["date", "benchmark_return"]
        ],
        on="date",
        how="inner"
    )

    merged = merged.dropna()

    slope, intercept, r, p, std = linregress(
        merged["benchmark_return"],
        merged["daily_return"]
    )

    beta = slope

    alpha = intercept * 252 * 100

    alpha_beta_data.append(
        [code, alpha, beta]
    )

alpha_beta_df = pd.DataFrame(
    alpha_beta_data,
    columns=[
        "amfi_code",
        "alpha_pct",
        "beta"
    ]
)

alpha_beta_df.head()

In [31]:
alpha_beta_df.sort_values(
    "alpha_pct",
    ascending=False
).head(10)

,amfi_code,alpha_pct,beta
21,119598,30.336965,-0.023196
39,149324,30.057878,0.011455
25,120505,29.263583,0.000549
36,148569,28.270368,0.018134
30,120843,27.330465,-0.022830
2,100033,27.195355,0.005104
34,148567,26.983751,0.023684
38,149323,26.598578,-0.002523
16,119094,26.076669,-0.066265
19,119551,23.201007,-0.031751


In [32]:
performance_df = performance_df.merge(
    alpha_beta_df,
    on="amfi_code"
)

performance_df.head()

,amfi_code,cagr_pct,sharpe_ratio,sortino_ratio,max_drawdown,cagr_rank,sharpe_rank,sortino_rank,drawdown_rank,fund_score,alpha_pct,beta
0,100016,2.635246,0.245276,0.427276,-0.247344,37.0,37.0,37.0,34.0,36.7,3.747581,-0.058268
1,100025,4.455091,1.097333,1.822432,-0.043083,36.0,26.0,26.0,4.0,27.8,4.281792,0.001158
2,100033,30.099704,1.436947,2.403193,-0.162172,7.0,13.0,13.0,20.0,11.3,27.195355,0.005104
3,101206,23.520489,1.473390,2.581215,-0.112916,12.0,12.0,11.0,9.0,11.5,21.399785,0.021086
4,101207,7.933121,0.414625,0.705169,-0.354469,27.0,36.0,36.0,38.0,32.6,10.897092,-0.065289


In [33]:
alpha_beta_df.to_csv(
    "../data/processed/alpha_beta.csv",
    index=False
)

In [34]:
performance_df.to_csv(
    "../data/processed/performance_metrics.csv",
    index=False
)

In [35]:
performance_df.sort_values(
    "fund_score"
).head(10)

,amfi_code,cagr_pct,sharpe_ratio,sortino_ratio,max_drawdown,cagr_rank,sharpe_rank,sortino_rank,drawdown_rank,fund_score,alpha_pct,beta
34,148567,30.949920,1.906241,3.139985,-0.112657,5.0,4.0,4.0,8.0,4.8,26.983751,0.023684
30,120843,30.883326,1.715884,3.104586,-0.129740,6.0,5.0,5.0,13.0,6.2,27.330465,-0.022830
36,148569,31.924486,1.602702,2.786282,-0.163967,4.0,7.0,7.0,21.0,7.2,28.270368,0.018134
25,120505,32.801599,1.517047,2.608779,-0.181885,1.0,9.0,9.0,25.0,7.4,29.263583,0.000549
19,119551,25.784921,1.681289,2.978156,-0.150124,10.0,6.0,6.0,17.0,8.7,23.201007,-0.031751
9,118632,24.031196,1.541077,2.635947,-0.174141,11.0,8.0,8.0,23.0,10.7,21.829398,-0.008354
38,149323,29.558105,1.498398,2.481754,-0.172481,8.0,10.0,12.0,22.0,10.8,26.598578,-0.002523
2,100033,30.099704,1.436947,2.403193,-0.162172,7.0,13.0,13.0,20.0,11.3,27.195355,0.005104
3,101206,23.520489,1.473390,2.581215,-0.112916,12.0,12.0,11.0,9.0,11.5,21.399785,0.021086
24,120504,23.277448,1.479051,2.601129,-0.125883,13.0,11.0,10.0,12.0,11.7,21.194827,0.016232
